# 바로핏 스쿼트 · Drive 데이터 · 1 대 4 비교
데이터: MyDrive의 **A Multi-View Raw Video Dataset of Seven Fitness Ex**로 시작하는 폴더.

원본 저장소의 공통 MediaPipe 추출기와 33×5 관절 규격을 사용합니다. 확인한 원본 커밋에는 스쿼트 엔진이 없어 이번 노트북에 **실험용 Python 엔진**을 추가했습니다. 플랭크 판정 기준은 사용하지 않습니다. 앱용 JavaScript 이식은 포함하지 않습니다.

결과: 기준 후보 1개 대 비교 후보 4개 사진+스켈레톤 GIF, 반복 횟수, 회차별 깊이, 몸통 기울기, 6개 각도 그래프, CSV/JSON, 짧은 피드백.

Good/Bad는 데이터 라벨이며 자동 정답 판정이 아닙니다. 아래 2D 임계값은 검증할 초기 설정입니다. 무릎 안쪽 붕괴·발뒤꿈치 들림·척추 중립은 이 코드에서 판단하지 않습니다. 일반 맨몸 스쿼트 측면 영상용이며 다른 변형/촬영 방향에서는 해석하지 마세요. 실제 영상 정확도는 별도 검증해야 합니다.


## 1. 저장소 및 실행 환경 준비


In [ ]:
import sys, subprocess
from pathlib import Path
SOURCE_URL = "https://github.com/Barofit-team/barofit-pose.git"
SOURCE_COMMIT = "a54ae2f1a8740388538db6829f95277f38e5d1c0"
REPO = Path("/content/barofit-pose-squat-reference")
if not REPO.exists():
    subprocess.run(["git", "clone", SOURCE_URL, str(REPO)], check=True)
if subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip() != SOURCE_URL:
    raise RuntimeError("기존 작업 폴더의 원격 주소가 다릅니다. REPO 경로를 바꾸세요.")
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)+"[video]", "pillow", "matplotlib"], check=True)
sys.path.insert(0, str(REPO))
print("준비 완료. 설치 직후 import 오류가 생기면 런타임을 다시 시작하고 1번부터 실행하세요.")


## 2. Drive 연결 · 데이터 폴더와 스쿼트 영상 검색
압축을 풀어 놓은 영상 폴더를 사용하세요. 폴더·파일명에 squat/스쿼트가 있는 영상만 후보로 표시합니다. 데이터 전체를 재귀 검색하므로 파일이 많으면 시간이 걸립니다. 여러 분류 폴더에 복제된 데이터가 있을 수 있어 동일 영상 중복 검사를 합니다.


In [ ]:
from google.colab import drive
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path
import unicodedata, re, json, hashlib, urllib.request
import numpy as np, pandas as pd, cv2
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Image as DisplayImage
from analysis.common.video import extract_video
from analysis.common.contract import json_safe

drive.mount("/content/drive")
PREFIX = "A Multi-View Raw Video Dataset of Seven Fitness Ex"
OUTPUT_ROOT = Path("/content/drive/MyDrive/Barofit/squat_results")
TARGET_FPS = 15
GIF_FPS = 5
GIF_SECONDS = 15
# 파일명으로 종목을 찾지 못할 때만 실제 스쿼트 하위 폴더의 전체 경로를 입력하세요.
SQUAT_SUBDIR = ""
def norm(value): return unicodedata.normalize("NFC",str(value)).casefold()
folders=sorted(p for p in Path("/content/drive/MyDrive").iterdir() if p.is_dir() and norm(p.name).startswith(norm(PREFIX)))
if not folders: raise FileNotFoundError("MyDrive 바로 아래에서 데이터 폴더를 찾지 못했습니다. 폴더명과 압축 해제를 확인하세요.")
if len(folders)>1:
    for i,p in enumerate(folders): print(i,p)
    n=int(input("데이터 폴더 번호: "))
    if not 0<=n<len(folders): raise ValueError("폴더 번호 오류")
    DATASET=folders[n]
else: DATASET=folders[0]
RUN_DIR=OUTPUT_ROOT/datetime.now(ZoneInfo("Asia/Seoul")).strftime("squat_%Y%m%d_%H%M%S_%f")
RUN_DIR.mkdir(parents=True,exist_ok=False)
CACHE_DIR=OUTPUT_ROOT/"cache"
search_root=Path(SQUAT_SUBDIR) if SQUAT_SUBDIR else DATASET
if not search_root.is_dir(): raise FileNotFoundError(search_root)
print("영상 검색 중:",search_root,flush=True)
videos=sorted(p for p in search_root.rglob("*") if p.is_file() and p.suffix.lower() in (".mp4",".mov",".avi",".mkv") and (SQUAT_SUBDIR or "squat" in norm(p.relative_to(DATASET)) or "스쿼트" in norm(p.relative_to(DATASET))))
if len(videos)<5: raise ValueError(f"스쿼트 후보 {len(videos)}개. 폴더 구조를 확인하고 SQUAT_SUBDIR에 스쿼트 폴더를 지정하세요.")
def label_hint(path):
    tokens=set(re.split(r"[^a-z]+",norm(path.relative_to(search_root))))
    good=bool(tokens & {"good","correct"});bad=bool(tokens & {"bad","incorrect"})
    return "ambiguous" if good and bad else "Good candidate" if good else "Bad candidate" if bad else "unknown"
video_table=pd.DataFrame([dict(number=i,label_hint=label_hint(p),path=str(p.relative_to(search_root))) for i,p in enumerate(videos)])
video_table.to_csv(RUN_DIR/"video_list.csv",index=False,encoding="utf-8-sig")
with pd.option_context("display.max_rows",200,"display.max_colwidth",150): display(video_table.head(200))
print("영상 수:",len(videos),"/ 전체 목록:",RUN_DIR/"video_list.csv")
print("결과:",RUN_DIR)


## 3. 측면 스쿼트 영상 5개 선택
위 목록에서 같은 촬영 방향의 기준 후보 1개와 비교 후보 4개를 선택하세요. 라벨이 없거나 축약되어 있으면 파일을 직접 확인합니다. 같은 사람의 다른 영상일 수 있으므로 4명 비교라고 자동 해석하지 않습니다.


In [ ]:
reference_index=int(input("기준 후보 번호: "))
comparison_indices=[int(x.strip()) for x in input("비교 후보 4개 번호 (쉼표 구분): ").split(",")]
selected=[reference_index]+comparison_indices
if len(selected)!=5 or len(set(selected))!=5 or any(i<0 or i>=len(videos) for i in selected):
    raise ValueError("중복 없는 영상 번호 5개를 선택하세요.")
CLIPS=[dict(id=f"case_{i}",role="Reference candidate" if i==0 else f"Comparison candidate {i}",path=videos[n],label=label_hint(videos[n])) for i,n in enumerate(selected)]
def sha_file(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()
if len({sha_file(c["path"]) for c in CLIPS})!=5: raise ValueError("내용이 같은 중복 영상이 있습니다. 다른 파일을 선택하세요.")
display(pd.DataFrame([dict(case=c["id"],role=c["role"],label=c["label"],file=c["path"].name) for c in CLIPS]))


## 4. 스쿼트 실험 엔진 정의
같은 쪽 관절을 사용합니다. 유효한 프레임 수가 많은 쪽을 영상 전체에서 고정 선택하므로 **오프라인 분석용**입니다. 실시간 좌우 선택 엔진은 아닙니다.

- 횟수: 서기(무릎 ≥155°) → 굽히기(≤125°) → 서기. 경계 상태를 0.12초 유지하고 최소 0.7초 간격을 둡니다.
- 깊이: 완료한 회차의 관측된 최소 무릎 각도가 100°보다 크면 `SQUAT_SHALLOW`. 이를 이유로 횟수를 취소하지 않습니다.
- 몸통 기울기: 화면 수직 대비 어깨–골반 각도 >45°가 0.6초 지속되면 `TRUNK_LEAN`. 40° 이하에서 해제합니다. 카메라 수평과 체형 영향을 받으며 허리 굽힘 자체를 측정하지 않습니다.
- 관절 누락 시 unknown. 0.25초 초과 간격이나 관절 누락은 미완료 회차를 끊습니다. 영상 시작이 굽힌 자세면 첫 불완전 회차는 세지 않습니다.

얕은 동작이 125°에 도달하지 않으면 이번 카운터에서는 누락됩니다. 임계값은 정확도 검증 전 초기값이며 정자세의 보편 기준이 아닙니다. 6개 각도는 좌우 무릎·고관절·팔꿈치입니다.


In [ ]:
ENGINE_SOURCE = '\nimport numpy as np\nIDS={"left":(11,13,15,23,25,27),"right":(12,14,16,24,26,28)}\nDEFAULT_CONFIG=dict(standing_deg=155.,bent_deg=125.,depth_deg=100.,state_hold_s=.12,min_rep_s=.7,max_gap_s=.25,trunk_enter_deg=45.,trunk_exit_deg=40.,trunk_hold_s=.6,min_quality=.5)\ndef angle(a,b,c):\n    u=a-b;v=c-b;den=np.linalg.norm(u)*np.linalg.norm(v)\n    return float(np.degrees(np.arccos(np.clip(np.dot(u,v)/den,-1,1)))) if den>1e-8 else None\n\ndef metrics(pose,dims,side,quality=.5):\n    p=np.asarray(pose,dtype=float);xy=p[:,:2]*np.asarray(dims)\n    s,e,w,h,k,a=IDS[side]\n    def valid(indices):\n        q=p[list(indices)]\n        return bool(np.isfinite(q).all() and (q[:,3:]>=quality).all() and ((q[:,:2]>=0)&(q[:,:2]<=1)).all())\n    knee=angle(xy[h],xy[k],xy[a]) if valid((h,k,a)) else None\n    hip=angle(xy[s],xy[h],xy[k]) if valid((s,h,k)) else None\n    elbow=angle(xy[s],xy[e],xy[w]) if valid((s,e,w)) else None\n    trunk=None\n    if valid((s,h)) and np.linalg.norm(xy[s]-xy[h])>1e-8:\n        trunk=float(np.degrees(np.arctan2(abs(xy[s,0]-xy[h,0]),xy[h,1]-xy[s,1])))\n    return dict(knee_deg=knee,hip_deg=hip,elbow_deg=elbow,trunk_deg=trunk)\n\nclass SquatAnalyzer:\n    def __init__(self,config=None):\n        self.c=dict(DEFAULT_CONFIG if config is None else config)\n        self.count=0;self.reps=[];self.last_t=None;self.last_rep=-float("inf")\n        self.reset()\n    def reset(self):\n        self.state="WAIT_STANDING";self.pending=None;self.pending_since=None\n        self.min_knee=None;self.start=None;self.lean_since=None;self.lean_warning=False\n    def step(self,t,knee,trunk):\n        if self.last_t is not None and t<=self.last_t: raise ValueError("시간은 단조 증가해야 합니다.")\n        if self.last_t is not None and t-self.last_t>self.c["max_gap_s"]:self.reset()\n        self.last_t=t\n        events=[];codes=[]\n        if knee is None or not np.isfinite(knee):\n            self.reset()\n            return dict(count=self.count,state="UNOBSERVABLE",codes=[],events=[],depth_state="unknown",trunk_state="unknown")\n        if self.state in ("READY","LOWERED"):\n            if self.start is None and knee<self.c["standing_deg"]: self.start=t\n            if self.start is not None:self.min_knee=knee if self.min_knee is None else min(knee,self.min_knee)\n        observed="standing" if knee>=self.c["standing_deg"] else "bent" if knee<=self.c["bent_deg"] else None\n        if observed is None:self.pending=None;self.pending_since=None\n        else:\n            if observed!=self.pending:self.pending=observed;self.pending_since=t\n            if t-self.pending_since>=self.c["state_hold_s"]:\n                if observed=="standing" and self.state=="WAIT_STANDING":self.state="READY"\n                elif observed=="bent" and self.state=="READY":self.state="LOWERED"\n                elif observed=="standing" and self.state=="LOWERED":\n                    if self.start is not None and t-self.last_rep>=self.c["min_rep_s"]:\n                        self.count+=1;self.last_rep=t\n                        shallow=self.min_knee>self.c["depth_deg"]\n                        event=dict(rep=self.count,start_s=self.start,end_s=t,min_knee_deg=self.min_knee,depth_state="warning" if shallow else "within_reference")\n                        self.reps.append(event);events.append(event)\n                        if shallow:codes.append("SQUAT_SHALLOW")\n                    self.state="READY";self.start=None;self.min_knee=None\n                elif observed=="standing" and self.state=="READY":self.start=None;self.min_knee=None\n        if trunk is None or not np.isfinite(trunk):\n            self.lean_since=None;self.lean_warning=False;trunk_state="unknown"\n        else:\n            if knee>=self.c["standing_deg"] or trunk<=self.c["trunk_exit_deg"]:\n                self.lean_since=None;self.lean_warning=False\n            elif trunk>self.c["trunk_enter_deg"] and knee<150:\n                if self.lean_since is None:self.lean_since=t\n                if t-self.lean_since>=self.c["trunk_hold_s"]:self.lean_warning=True\n            elif not self.lean_warning:self.lean_since=None\n            trunk_state="warning" if self.lean_warning else "pending_feedback" if self.lean_since is not None else "within_reference"\n            if self.lean_warning:codes.append("TRUNK_LEAN")\n        return dict(count=self.count,state=self.state,codes=codes,events=events,depth_state=events[-1]["depth_state"] if events else "not_evaluated",trunk_state=trunk_state)\n'
ENGINE_DIR = RUN_DIR / "export" / "analysis" / "squat"
ENGINE_DIR.mkdir(parents=True,exist_ok=True)
(ENGINE_DIR/"experimental.py").write_text(ENGINE_SOURCE,encoding="utf-8")
exec(compile(ENGINE_SOURCE,"experimental.py","exec"))
CONFIG=dict(DEFAULT_CONFIG)
print(CONFIG)


## 5. 원본 영상에서 MediaPipe 관절 추출
전체 영상을 분석합니다. 긴 영상은 미리 잘라 사용하세요. 캐시는 영상·모델 해시로 재사용합니다.


In [ ]:
MODEL = Path("/content/models/pose_landmarker_full.task")
MODEL.parent.mkdir(exist_ok=True)
if not MODEL.exists():
    urllib.request.urlretrieve("https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task", MODEL)
DATA = {}
for clip in CLIPS:
    print("추출 중:", clip["path"].name, flush=True)
    arrays, info = extract_video(clip["path"], MODEL, CACHE_DIR, target_fps=TARGET_FPS)
    DATA[clip["id"]] = {"arrays": arrays, "info": info}
    observed = np.isfinite(arrays["poses"][:, :, :2]).all(axis=(1,2)).sum()
    print(f"샘플 {len(arrays['times'])}개 / 전 좌표 유한 프레임 {observed}개")
print("추출 완료")


## 6. 5개 영상 분석 · 횟수·회차·피드백 저장
기준 영상과의 차이로 오류를 억지 생성하지 않고 각 영상에 같은 규칙을 독립 적용합니다. 인식 불가와 경고 없음은 구분합니다. API 키는 필요하지 않습니다.


In [ ]:
FEEDBACK={"SQUAT_SHALLOW":"통증 없는 범위에서 무릎을 조금 더 굽혀보세요.","TRUNK_LEAN":"상체가 과하게 숙여지지 않도록 조절하세요."}
SIDE_OVERRIDES={}  # 예: {"case_0":"left"}
ALL_ROWS=[];ALL_REPS=[];SUMMARY=[]
for clip in CLIPS:
    cid=clip["id"];arrays=DATA[cid]["arrays"]
    candidates={side:[metrics(p,d,side,CONFIG["min_quality"]) for p,d in zip(arrays["poses"],arrays["dims"])] for side in IDS}
    side=SIDE_OVERRIDES.get(cid) or max(IDS,key=lambda s:sum(m["knee_deg"] is not None for m in candidates[s]))
    if side not in IDS: raise ValueError("좌우 설정은 left/right만 가능합니다.")
    engine=SquatAnalyzer(CONFIG);outputs=[];rows=[]
    for n,(t,index) in enumerate(zip(arrays["times"],arrays["indices"])):
        m=candidates[side][n]
        r=engine.step(float(t),m["knee_deg"],m["trunk_deg"])
        output=dict(schemaVersion="1.0",exercise="squat",engineVersion="squat-offline-experimental-0.1",timestampMs=float(t)*1000,side=side,tracking="tracking_unavailable" if r["state"]=="UNOBSERVABLE" else "tracking",ignored=False,holdSeconds=None,items=dict(depth=r["depth_state"],trunk=r["trunk_state"]),metrics=m,**r)
        outputs.append(output)
        row=dict(case=cid,source_frame=int(index),time_s=float(t),side=side,count=r["count"],state=r["state"],codes="|".join(r["codes"]),depth_state=r["depth_state"],trunk_state=r["trunk_state"],**m)
        for s in IDS:
            for key in ("knee_deg","hip_deg","elbow_deg"):row[s+"_"+key]=candidates[s][n][key]
        rows.append(row)
    table=pd.DataFrame(rows);DATA[cid].update(outputs=outputs,table=table,side=side)
    out=RUN_DIR/cid;out.mkdir(exist_ok=True)
    table.to_csv(out/"frames.csv",index=False,encoding="utf-8-sig")
    (out/"frames.jsonl").write_text("".join(json.dumps(json_safe(r),ensure_ascii=False,allow_nan=False)+"\n" for r in outputs),encoding="utf-8")
    for rep in engine.reps:ALL_REPS.append(dict(case=cid,**rep))
    SUMMARY.append(dict(case=cid,dataset_label_hint=clip["label"],side=side,count=engine.count,depth_warning_reps=sum(r["depth_state"]=="warning" for r in engine.reps),tracked_frame_pct=100*sum(r["state"]!="UNOBSERVABLE" for r in outputs)/len(outputs)))
    ALL_ROWS.extend(rows)
summary=pd.DataFrame(SUMMARY)
summary.to_csv(RUN_DIR/"summary.csv",index=False,encoding="utf-8-sig")
repetitions=pd.DataFrame(ALL_REPS,columns=["case","rep","start_s","end_s","min_knee_deg","depth_state"])
repetitions.to_csv(RUN_DIR/"repetitions.csv",index=False,encoding="utf-8-sig")
pd.DataFrame(ALL_ROWS).to_csv(RUN_DIR/"all_frames.csv",index=False,encoding="utf-8-sig")
display(summary);display(repetitions)
for clip in CLIPS:
    codes=sorted({c for r in DATA[clip["id"]]["outputs"] for c in r["codes"]})
    print(clip["id"],[FEEDBACK[c] for c in codes] or ["검출된 경고 없음. 추적률과 실제 영상을 함께 확인하세요."])


## 7. 사진+스켈레톤 GIF 4개 · 동작별 카운트
왼쪽 기준 후보, 오른쪽 비교 후보입니다. 공통 시작 구간을 같은 경과 시간으로 보여주며 동작 단계 동기화는 아닙니다. GIF 시작 15초만 표시하지만 분석/횟수는 영상 전체입니다. 깊이 피드백은 복귀 후 회차가 완료되어야 나오며 1.2초 표시합니다. 반복 프레임은 재카운트하지 않습니다.


In [ ]:
font_path = Path("/content/NotoSansKR.ttf")
try:
    if not font_path.exists():
        urllib.request.urlretrieve("https://raw.githubusercontent.com/google/fonts/main/ofl/notosanskr/NotoSansKR%5Bwght%5D.ttf", font_path)
    FONT = ImageFont.truetype(str(font_path), 16)
except Exception:
    FONT = ImageFont.truetype("DejaVuSans.ttf", 14)
EDGES = [(11,13),(13,15),(12,14),(14,16),(11,12),(11,23),(12,24),(23,24),(23,25),(25,27),(24,26),(26,28)]

def panel(frame, clip, n):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    photo = Image.fromarray(rgb)
    photo.thumbnail((480, 300))
    canvas = Image.new("RGB", (500, 530), "#101827")
    x0, y0 = (500-photo.width)//2, 60+(300-photo.height)//2
    canvas.paste(photo, (x0,y0))
    draw = ImageDraw.Draw(canvas)
    arrays = DATA[clip["id"]]["arrays"]
    p = arrays["poses"][n]
    result = DATA[clip["id"]]["outputs"][n]
    def valid(i):
        return bool(np.isfinite(p[i]).all() and min(p[i,3:]) >= .5 and ((p[i,:2]>=0)&(p[i,:2]<=1)).all())
    def xy(i): return (x0+float(p[i,0])*photo.width, y0+float(p[i,1])*photo.height)
    for a,b in EDGES:
        if valid(a) and valid(b): draw.line([xy(a),xy(b)], fill="#51d5ff", width=3)
    for i in sorted({i for edge in EDGES for i in edge}):
        if valid(i):
            x,y=xy(i);draw.ellipse((x-3,y-3,x+3,y+3), fill="#ffe66d")
    draw.text((10,8), clip["role"], font=FONT, fill="white")
    draw.text((10,32), f"t={arrays['times'][n]:.2f}s / side={side} / {result['tracking']}", font=FONT, fill="white")
    draw.text((10,366), f"Reps: {result['count']} / {result['state']}", font=FONT, fill="#9de6b4")
    m=result["metrics"]
    for j,key in enumerate(("knee_deg","trunk_deg")):
        value=m[key]
        text=f"{key}: {value:.1f}" if value is not None else f"{key}: N/A"
        draw.text((10,390+j*22),text,font=FONT,fill="white")
    current_t=arrays["times"][n]
    codes=set(result["codes"])
    if result["tracking"]!="tracking_unavailable":
        for j in range(n,-1,-1):
            if current_t-arrays["times"][j]>1.2:break
            if "SQUAT_SHALLOW" in DATA[clip["id"]]["outputs"][j]["codes"]:codes.add("SQUAT_SHALLOW")
    feedback=[FEEDBACK[c] for c in sorted(codes)]
    if result["tracking"]=="tracking_unavailable":feedback=["관절 인식 불가: 자세 판정 보류"]
    if not feedback:feedback=["검출된 경고 없음 / 깊이는 완료한 회차에서 평가"]
    for j,line in enumerate(feedback[:3]):draw.text((10,448+j*23),line,font=FONT,fill="white")
    return canvas

for other in CLIPS[1:]:
    pair=[CLIPS[0],other]
    caps=[cv2.VideoCapture(str(c["path"])) for c in pair]
    duration=min(DATA[c["id"]]["arrays"]["times"][-1] for c in pair)
    grid=np.arange(0,min(GIF_SECONDS,duration),1/GIF_FPS)
    frames=[]
    try:
        for t in grid:
            panels=[]
            for cap,clip in zip(caps,pair):
                arrays=DATA[clip["id"]]["arrays"]
                n=int(np.argmin(abs(arrays["times"]-t)))
                cap.set(cv2.CAP_PROP_POS_FRAMES,int(arrays["indices"][n]))
                ok,frame=cap.read()
                if not ok: raise RuntimeError(f"GIF 원본 프레임 읽기 실패: {clip['path'].name}")
                panels.append(panel(frame,clip,n))
            merged=Image.new("RGB",(1010,530),"white")
            merged.paste(panels[0],(0,0));merged.paste(panels[1],(510,0));frames.append(merged)
    finally:
        for cap in caps: cap.release()
    if not frames: raise RuntimeError("GIF를 만들 수 있을 만큼 긴 영상이 없습니다.")
    path=RUN_DIR/f"reference_vs_{other['id']}.gif"
    frames[0].save(path,save_all=True,append_images=frames[1:],duration=round(1000/GIF_FPS),loop=0)
    display(DisplayImage(filename=str(path)))
    print(path.name)


## 8. 합쳐진 그래프와 영상별 6개 각도 그래프
무릎 각도는 스쿼트에서 반복적으로 줄고 커지는 것이 자연스럽습니다. 평평함만으로 정자세를 판단하지 않습니다. 몸통 기울기는 화면 수직 기준이며 팔꿈치·고관절은 기록용입니다. 깊이 점선은 최저점 평가용이지 모든 프레임에서 만족할 선이 아닙니다.


In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(2,1,figsize=(12,8),constrained_layout=True)
for clip in CLIPS:
    table=DATA[clip["id"]]["table"]
    axes[0].plot(table.time_s,table.knee_deg,label=clip["id"])
    axes[1].plot(table.time_s,table.trunk_deg,label=clip["id"])
for v in (CONFIG["standing_deg"],CONFIG["bent_deg"],CONFIG["depth_deg"]):axes[0].axhline(v,color="black",linestyle="--",alpha=.4)
axes[1].axhline(CONFIG["trunk_enter_deg"],color="black",linestyle="--",alpha=.4)
for ax,title in zip(axes,["Selected-side knee angle (deg)","Trunk vertical inclination (deg)"]):
    ax.set_title(title);ax.set_xlabel("Time (s)");ax.legend();ax.grid(alpha=.2)
fig.savefig(RUN_DIR/"combined_metrics.png",dpi=160);plt.show()
for clip in CLIPS:
    table=DATA[clip["id"]]["table"]
    fig,axes=plt.subplots(3,2,figsize=(12,10),constrained_layout=True)
    for row,key in enumerate(("knee_deg","hip_deg","elbow_deg")):
        for col,side in enumerate(("left","right")):
            ax=axes[row,col];ax.plot(table.time_s,table[side+"_"+key]);ax.set_title(side+" "+key);ax.set_xlabel("Time (s)");ax.set_ylabel("Degrees");ax.grid(alpha=.2)
            if key=="knee_deg":ax.axhline(CONFIG["depth_deg"],color="black",linestyle="--",alpha=.4)
    fig.suptitle(clip["id"])
    fig.savefig(RUN_DIR/f"{clip['id']}_six_angles.png",dpi=140);plt.show()


## 9. 설정·원본 출처 저장
원본 규칙을새 규칙처럼 표기하지 않으며 스쿼트는 이번 실험 구현임을 기록합니다. 모델/영상/관절 결과는 Git에서 제외합니다.


In [ ]:
settings=dict(exercise="squat",implementation="offline-experimental-python",source_url=SOURCE_URL,source_commit=SOURCE_COMMIT,config=CONFIG,target_fps=TARGET_FPS,gif_fps=GIF_FPS,gif_seconds=GIF_SECONDS,side_overrides=SIDE_OVERRIDES,clips=[dict(case=c["id"],role=c["role"],label_hint=c["label"],source_sha256=DATA[c["id"]]["info"]["source_sha256"]) for c in CLIPS],limitations=["측면 맨몸 스쿼트", "오프라인 고정 좌우 선택", "임계값은 미검증 초기 설정", "변형 스쿼트/무릎 안쪽 붕괴/발뒤꿈치/척추중립 판단 미지원", "125도에 도달하지 않은 얕은 움직임 카운트 누락 가능", "실제 영상 정확도 미검증"])
(RUN_DIR/"settings.json").write_text(json.dumps(settings,ensure_ascii=False,indent=2),encoding="utf-8")
print("저장 완료:",RUN_DIR)
print("GIF 4개, 통합 그래프, 영상별 6개 각도 그래프, 프레임·회차·요약 CSV, 설정 JSON")
print("엔진 모듈:",ENGINE_DIR/"experimental.py")


## 10. Git 반영
원본 README에서 스쿼트는 `analysis/squat/`에 추가하도록 안내합니다. 이번 노트북은 공통 추출기를 고정 커밋에서 가져오고, 스쿼트 엔진은 4번 셀에서 정의하여 `결과폴더/export/analysis/squat/experimental.py`로 내보냅니다.

팀 저장소에 기여하려면 `feat/squat-colab` 브랜치를 만들고 노트북을 `examples/notebooks/barofit_squat_comparison.ipynb`, 내보낸 모듈을 `analysis/squat/experimental.py`에 넣으세요. 규칙 변경은 모듈을 원본으로 유지하고 노트북을 모듈 import 방식으로 전환해야 합니다. 아직 원격 저장소는 변경하지 않았습니다.

개인 레포가 필요하면 원본을 Fork하고 그곳에서 같은 작업을 하세요. 기존 README와 출처를 유지합니다.

```bash
python -m pip install -e .
python -m unittest discover -s tests/python -v
npm test
```
위 명령은 기존 엔진 회귀 검증입니다. 스쿼트 신규 로직은 아래 합성 점검과 별도로 실제 영상 수동 카운트·라벨과 비교 검증해야 합니다. 합성 점검 통과를 자세 정확도로 표현하지 마세요.

Git에는 출력이 제거된 노트북·엔진·테스트·규칙 문서만 올립니다. 원본 영상/NPZ/GIF/CSV/모델/개인 경로는 제외합니다. `docs/RULES.md`에 4번 셀의 수식·임계값·한계를, README에 Python 실험 단계 및 JS 미구현을 기록하세요. PR에 결과와 검증 한계를 적습니다. 데이터·모델의 원 출처 권리는 그대로 적용됩니다.


## 11. 합성 신호 점검 (영상 처리와 독립)
횟수와 깊이 평가 분리, 인식 누락, 정지, 역순 입력을 점검합니다. 실제 자세 정확도 검증은 아닙니다.


In [ ]:
def feed(engine,values,start=0):
    t=start
    for knee in values:
        engine.step(t,knee,10. if knee is not None else None)
        t+=.05
    return t
engine=SquatAnalyzer(CONFIG)
t=feed(engine,[170.]*8+[120.]*8+[170.]*8)
assert engine.count==1 and engine.reps[0]["depth_state"]=="warning"
t=feed(engine,[90.]*8+[170.]*8,t)
assert engine.count==2 and engine.reps[1]["depth_state"]=="within_reference"
engine=SquatAnalyzer(CONFIG)
feed(engine,[170.]*8+[90.]*8+[None]+[170.]*8)
assert engine.count==0
engine=SquatAnalyzer(CONFIG)
feed(engine,[170.]*100)
assert engine.count==0
try:engine.step(0,170.,10.)
except ValueError:pass
else:raise AssertionError("역순 입력이 허용되었습니다.")
print("PASS: 합성 횟수·깊이 분리, 누락 시 미완성 동작 중단, 정지·역순 입력")
